# 04 — ¿El clustering identifica los escenarios?

El notebook 03 entrenó un detector **solo con benigno** y mostró que aprende regímenes de operación, no escenarios. Acá atacamos la pregunta que quedó abierta y que el profesor sugirió: **¿un método no supervisado puede identificar de qué escenario viene una fila?** Aunque sea con unos pocos.

**Recorrido:**

1. Elegir cuatro escenarios, uno por familia de ataque.
2. Definir la métrica honesta (ARI) y por qué la pureza sola engaña.
3. Clusterizar con K-Means (k = número de escenarios) sobre dos entradas: niveles absolutos y dinámica local.
4. Comparar la tabla de contingencia y leer el resultado.

La trampa que hay que evitar desde el arranque: cada escenario es una corrida de ~30 minutos en un momento distinto del día. Si el detector separa escenarios por el **nivel de operación** (irradiancia, carga de base), no aprendió nada físico: está leyendo el reloj. Por eso medimos las dos cosas por separado.

## Antes de empezar: si lo abrís en Google Colab

Este notebook vive en el repo y lee `datasets/` y `src/` desde la raíz del proyecto. En Colab esos archivos no existen, así que la celda de abajo clona el repo en `/content/data4cyber` y se para ahí.

1. En Colab: *Archivo → Abrir notebook → GitHub*, pegá la URL del repo y elegí este notebook.
2. Ejecutá la primera celda. Clona el repo (una sola vez por sesión) y queda listo para correr.

Fuera de Colab la celda no hace nada.

In [ ]:
# --- Preparar el entorno: solo hace algo si estás en Google Colab ---
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/maxiricalde/data4cyber.git"
DESTINO = Path("/content/data4cyber")

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    if not (DESTINO / "src" / "data.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(DESTINO)], check=True)
    os.chdir(DESTINO)
    print("proyecto en:", os.getcwd())
else:
    print("corriendo local: no hay nada que preparar")

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# El notebook vive en notebooks/, pero los datasets y src/ están en la raíz del proyecto.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").exists())
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from src.data import build_matrices
from src.features import detrend

pd.set_option("display.width", 140)
print("raíz del proyecto:", RAIZ)

## 0. Cuatro escenarios, uno por familia

Para no depender de una elección arbitraria de archivos, tomamos un representante de cada familia de ataque. La etiqueta que el clustering debería recuperar es el **escenario** (S0, S1, S3, S6), no el ataque: queremos saber si el método distingue *de dónde viene* la fila.

| Escenario | Familia | Tipo |
|---|---|---|
| S0 | benigno | operación normal |
| S1 | `modbus_pv` | Industroyer contra el inversor PV |
| S3 | `arp` | ARP/MitM sobre el medidor BSS |
| S6 | `mqtt` | compromiso del broker de precio |

In [ ]:
ESCENARIOS = ("S0", "S1", "S3", "S6")

frames = pd.concat(
    [pd.read_csv(f"datasets/dataset{nombre}.csv").assign(scenario=nombre) for nombre in ESCENARIOS],
    ignore_index=True,
)

m = build_matrices(frames)
X_niveles = m.X
X_dinamica = detrend(X_niveles, m.scenario, method="range", window=120)
escenario = m.scenario.to_numpy()

print("filas:", len(X_niveles), "| columnas de entrada:", X_niveles.shape[1])
print(pd.Series(escenario).value_counts().sort_index().to_string())

## 1. La métrica honesta: ARI, no pureza

Hay dos formas de decir "el cluster encontró el escenario":

- **Pureza**: qué fracción de cada cluster pertenece a su escenario mayoritario. Sube sola cuando los clusters son chicos y numerosos: si cada fila es su propio cluster, la pureza da 1. Por eso engaña.
- **ARI** (Adjusted Rand Index): compara el agrupamiento contra la etiqueta real y **descarta el acuerdo por azar**. Vale 0 para un agrupamiento aleatorio y 1 para una coincidencia perfecta. Es el número que usamos.

También fijamos `k = 4`, el número real de escenarios. Es la prueba justa: "dado que hay cuatro, ¿podés recuperar los cuatro?" No dejamos que el modelo invente un k cómodo.

In [ ]:
SEMILLA = 42

def preparar(X):
    pipe = Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=SEMILLA)),
    ])
    return pipe.fit_transform(X), pipe


def pureza(etiquetas, verdad):
    """Fracción de filas cuyo cluster está dominado por su escenario mayoritario."""
    tab = pd.crosstab(etiquetas, verdad)
    return float(tab.max(axis=1).sum() / tab.values.sum())


def evaluar(nombre, Z, verdad, k=4):
    modelo = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Z)
    ari = adjusted_rand_score(verdad, modelo.labels_)
    return modelo, ari, pureza(modelo.labels_, verdad), Z.shape[1]


resultados = {}
for nombre, X in (("niveles", X_niveles), ("dinámica", X_dinamica)):
    Z, _ = preparar(X)
    _, ari, pur, ncomp = evaluar(nombre, Z, escenario)
    resultados[nombre] = {"ARI": ari, "pureza": pur, "componentes PCA": ncomp}
    print(f"{nombre:9s} -> ARI {ari:.3f} | pureza {pur:.3f} | PCA {ncomp} comp.")

print()
print(pd.DataFrame(resultados).T.round(3).to_string())

## 2. La tabla de contingencia

La ARI resume, pero la tabla muestra *dónde* se separa y dónde se mezcla. Cada fila es un cluster y cada columna un escenario; los números son filas.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.6))

for eje, (nombre, X) in zip(ejes, (("niveles", X_niveles), ("dinámica", X_dinamica))):
    Z, _ = preparar(X)
    km = KMeans(n_clusters=4, n_init=10, random_state=SEMILLA).fit(Z)
    tab = pd.crosstab(km.labels_, escenario)
    tab.index.name = "cluster"
    tab.columns.name = "escenario"
    print(f"--- {nombre} ---")
    print(tab.to_string())
    print()

    # proporción de cada escenario dentro de cada cluster (visual)
    prop = tab.div(tab.sum(axis=1), axis=0)
    prop.plot(kind="bar", stacked=True, ax=eje, legend=False)
    eje.set_title(nombre)
    eje.set_ylabel("proporción del cluster")
    eje.set_xlabel("cluster")

plt.tight_layout()
plt.show()

## 3. Qué dice esto

**Ni los niveles ni la dinámica separan los escenarios.**

- Con **dinámica** (la forma local, sin el nivel) la ARI da ~0,02: es azar. Un detector que solo mira cómo se mueve la señal en los últimos dos minutos no distingue un escenario de otro.
- Con **niveles** (donde vive el reloj, vía irradiancia y carga de base) la ARI da ~0,12: apenas por encima del azar. La tabla muestra que solo **S1** (el ataque de mediodía contra el PV) forma un cluster parcial; los otros tres se mezclan.

Que los niveles no separen tampoco tiene una razón concreta y verificable: las irradiancias medias son S0 ≈ 643, S1 ≈ 434, S3 ≈ 428 y S6 ≈ 425 W/m². Solo S0 destaca; S1, S3 y S6 corren a horas muy distintas del día (11:49, 17:15 y 14:48 local) pero la nubosidad las deja con un nivel parecido. **El reloj no se traduce en una separación limpia de escenarios.**

Esto refuerza la conclusión de los notebooks 02 y 03: la identidad del escenario no es una señal que un método no supervisado recupere por nivel ni por forma. Lo que el dataset sí permite —y es lo que explota el notebook 03— es aprender la forma del tráfico **benigno dentro de un despliegue** y marcar lo raro respecto de esa línea de base.

## Cierre

Lo que este notebook establece:

1. Con cuatro escenarios (uno por familia), el clustering **no los identifica**: ARI ≈ 0,12 con niveles y ≈ 0,02 con dinámica.
2. La pureza sola sería engañosa (fragmentar en muchos clusters la infla); la ARI es la métrica correcta.
3. El nivel de operación no es un atajo limpio para el escenario: la nubosidad mezcla horas distintas del día.
4. El resultado confirma la lectura conjunta: no hay una señal *entre* escenarios que un detector no supervisado recupere; la señal utilizable es *dentro* de un despliegue.